# AvoCast — Forecasting U.S. Avocado Prices with Prophet

**Author:** Marvin Neumann · IU14156706  
**Course:** DLBAIBEPAIPC01 — Project: AI Product Commercialisation  
**Region:** SanDiego (San Diego metropolitan area)  
**Type:** conventional avocados

---

## Task 3 — Load and Explore the Data

**Brief requirement (Task brief, p. 4):**
> *"Load avocado.csv from Kaggle using pandas. Review columns such as date, average price, type, and region. Choose one region (e.g., TotalUS)."*

Each step below maps directly to one of those instructions.

### Cell 1 — Setup

We import the libraries we need:
- `pandas` — handles tabular data (the CSV)
- `matplotlib.pyplot` — plotting library

We also configure the plot style for cleaner figures.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

### Load the dataset

We load `avocado.csv` with pandas into a DataFrame. `df.head()` confirms the file parsed correctly and shows what one row looks like — one weekly observation for one region and one avocado type.

In [ ]:
df = pd.read_csv('../data/avocado.csv')
df.head()

### Review the columns

The four columns named in the brief are all present: `Date` (weekly timestamp), `AveragePrice` (our forecast target in USD), `type` (`conventional` or `organic`), and `region` (one of 54 US regions). The other columns describe sales volume by PLU code and bag size — useful context, but not needed for a price forecast.

In [ ]:
print('Shape (rows, columns):', df.shape)
print('Columns:', list(df.columns))
print('Date range:', df['Date'].min(), 'to', df['Date'].max())
print('Types available:', df['type'].unique())
print('Regions available:', df['region'].nunique())

### Choose one region: SanDiego, conventional

The brief allows any single region. We pick **SanDiego** (a real metropolitan market makes the business translation in Task 7 more concrete than the aggregate `TotalUS`) and restrict to **conventional** avocados, which dominate sales volume and show clearer seasonal patterns than the organic premium segment. Full rationale in `decisions/002-user-segment.md`.

We also convert `Date` to a real datetime and sort chronologically so the series is ready for time-series modelling.

In [ ]:
sd = df[(df['region'] == 'SanDiego') & (df['type'] == 'conventional')].copy()
sd['Date'] = pd.to_datetime(sd['Date'])
sd = sd.sort_values('Date').reset_index(drop=True)

print('Filtered shape:', sd.shape)
print('Date range:', sd['Date'].min().date(), 'to', sd['Date'].max().date())
sd[['Date', 'AveragePrice', 'Total Volume', 'type', 'region']].head()

### Cell 5 — Plot historical prices

A simple line chart of `AveragePrice` over time, so we can visually inspect:
- Overall trend (going up, down, or flat?)
- Seasonal patterns (do prices spike at certain times of year?)
- Outliers / unusual events

These observations will inform the model configuration in Task 5 (which seasonalities to enable, which holidays to add).

In [ ]:
fig, ax = plt.subplots()
ax.plot(sd['Date'], sd['AveragePrice'], color='#2f9e44', linewidth=1.2)
ax.set_title('San Diego — Conventional Avocado Weekly Average Price', loc='left', fontsize=13, weight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Average Price ($)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Task 4 — Preprocess for Prophet

Prophet expects exactly two columns with fixed names: `ds` (datestamp) and `y` (the value to forecast). We rename, sort chronologically, and check that the weekly cadence has no gaps — Prophet handles missing weeks badly, so a clean weekly index is important before training.

In [ ]:
pp = (sd[['Date', 'AveragePrice']]
        .rename(columns={'Date': 'ds', 'AveragePrice': 'y'})
        .sort_values('ds')
        .reset_index(drop=True))

gaps = pp['ds'].diff().dropna()
print('Unique gaps between observations:', gaps.unique())
print('Expected: only 7 days (weekly).')

if not (gaps == pd.Timedelta(days=7)).all():
    pp = (pp.set_index('ds')
            .asfreq('W-SUN')
            .interpolate(method='linear')
            .reset_index())
    print('Gaps filled by linear interpolation.')

print('\nFinal shape:', pp.shape)
pp.head()

## Task 5 — Train/test split and fit Prophet

We hold out the **last 20%** of the weeks as a test set so the model is evaluated on data it has never seen. Prophet is trained on the first 80% with **yearly seasonality** (avocado prices follow a clear annual pattern) and a custom set of **avocado-relevant US holidays** — Super Bowl (guacamole peak), Cinco de Mayo (Mexican-heritage demand), Thanksgiving and Christmas. We deliberately skip the generic full US-holiday list because most federal holidays (Labor Day, Veterans Day, MLK) don't shift avocado demand and would only add noise.

Forecast accuracy is reported as **MAPE** — the average percentage error between predicted and actual price. Below 10% is generally considered a strong forecast.

In [ ]:
from prophet import Prophet
import numpy as np

split = int(len(pp) * 0.8)
train, test = pp.iloc[:split].copy(), pp.iloc[split:].copy()
print(f'Train: {len(train)} weeks ({train["ds"].min().date()} to {train["ds"].max().date()})')
print(f'Test:  {len(test)} weeks ({test["ds"].min().date()}  to {test["ds"].max().date()})')

holidays = pd.DataFrame([
    {'holiday': 'super_bowl',    'ds': '2015-02-01'},
    {'holiday': 'super_bowl',    'ds': '2016-02-07'},
    {'holiday': 'super_bowl',    'ds': '2017-02-05'},
    {'holiday': 'super_bowl',    'ds': '2018-02-04'},
    {'holiday': 'cinco_de_mayo', 'ds': '2015-05-05'},
    {'holiday': 'cinco_de_mayo', 'ds': '2016-05-05'},
    {'holiday': 'cinco_de_mayo', 'ds': '2017-05-05'},
    {'holiday': 'cinco_de_mayo', 'ds': '2018-05-05'},
    {'holiday': 'thanksgiving',  'ds': '2015-11-26'},
    {'holiday': 'thanksgiving',  'ds': '2016-11-24'},
    {'holiday': 'thanksgiving',  'ds': '2017-11-23'},
    {'holiday': 'christmas',     'ds': '2015-12-25'},
    {'holiday': 'christmas',     'ds': '2016-12-25'},
    {'holiday': 'christmas',     'ds': '2017-12-25'},
])
holidays['ds'] = pd.to_datetime(holidays['ds'])
holidays['lower_window'] = -7
holidays['upper_window'] = 7

m = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    holidays=holidays,
)
m.fit(train)

future = m.make_future_dataframe(periods=len(test), freq='W-SUN')
forecast = m.predict(future)

yhat = forecast.iloc[-len(test):]['yhat'].values
y_true = test['y'].values
mae = np.mean(np.abs(y_true - yhat))
mape = np.mean(np.abs((y_true - yhat) / y_true)) * 100

print(f'\nMAE:  ${mae:.3f}')
print(f'MAPE: {mape:.2f}%')

## Task 6 — Visualize and explain

Three diagnostic plots help us understand what the model has actually learned: the **forecast itself** (with the held-out test points overlaid), the **components** (trend, yearly seasonality, holiday effects shown separately), and the **change points** Prophet detected — the weeks where the underlying trend shifted direction.

In [ ]:
from prophet.plot import add_changepoints_to_plot

fig1 = m.plot(forecast)
ax = fig1.gca()
ax.scatter(test['ds'], test['y'], color='#e03131', s=18, label='Actual (test set)', zorder=5)
ax.set_title('Forecast vs. Actual — SanDiego conventional avocado price', loc='left', weight='bold')
ax.set_xlabel('Date'); ax.set_ylabel('Average Price ($)')
ax.legend(loc='upper left')
add_changepoints_to_plot(ax, m, forecast)
plt.tight_layout()
fig1.savefig('../assets/forecast.png', dpi=120, bbox_inches='tight')
plt.show()

fig2 = m.plot_components(forecast)
fig2.savefig('../assets/components.png', dpi=120, bbox_inches='tight')
plt.show()

### How Prophet works — in plain language

Prophet treats a time series as a **sum of four ingredients**:

> **price = trend + yearly pattern + holiday effects + noise**

- **Trend** is the long-run direction — is the average price drifting up, down, or staying flat over the years? Prophet draws a flexible line through the data and automatically picks a few "change points" where the slope shifts (e.g. a 2017 supply shock).
- **Yearly pattern** is the wave that repeats every 12 months — avocado prices tend to climb into summer and dip in winter. Prophet fits a smooth curve to this seasonal cycle.
- **Holiday effects** are short bumps around specific dates we hand it (Super Bowl, Cinco de Mayo, Thanksgiving, Christmas). Prophet learns: *"on this week each year, prices behave differently than the trend + season would predict."*
- **Noise** is everything left over — random week-to-week wiggle the model can't explain.

When forecasting, Prophet just **adds these four pieces back together** for any future date. That's why it's called an *additive* model. The big advantage: each piece can be visualised separately (the components plot above), so the forecast is interpretable — you see *why* the model predicts what it predicts, not just *what*.

The change points marked on the forecast plot are the moments where Prophet decided the trend slope changed. They are useful business signals on their own: each one usually corresponds to a real-world supply or demand shift worth investigating.

## Task 7 — From forecast to retailer actions

A forecast is only useful if it changes a decision. The components plot gives us four signals that translate directly into retailer playbooks. A high-demand week does not automatically mean "raise shelf prices". In avocado retail, Super Bowl and Cinco de Mayo often work as promotion windows: volume goes up, but average per-unit price can go down because stores discount heavily to bring people in.

**1. Plan promotions, not margin lifts, around Super Bowl and Cinco de Mayo**
The model shows negative average-price effects for those weeks (about -$0.11 to -$0.12). That does not mean demand is weak. It means retailers likely use avocado as a traffic product: lower price, more units sold, bigger baskets with chips, salsa, tacos, or beer. Order 2-3 weeks ahead and secure enough volume, but do not assume the best move is a higher shelf price.

**2. Treat Thanksgiving as the cleaner margin window**
Thanksgiving shows the clearest positive holiday price effect. That makes it a better moment to hold price closer to the seasonal trend instead of using deep discounts. The action is still to prepare stock early, but the commercial goal is margin protection rather than pure traffic.

**3. Use the Christmas / New Year dip as a chosen promo window**
Mid-December to early January, prices tend to soften. Christmas is not really an avocado moment in the U.S. food calendar, but that creates an opening. Run recipe cards, bundle with tortilla chips, and use promotions to create demand instead of waiting for it.

**4. Use the forecast for supplier contracts, not just shelf decisions**
For growers and wholesalers, the forecast is less about changing shelf price and more about timing volume and contracts. A grower cannot simply lift prices at will, but can plan harvest timing and expected volume. A wholesaler can negotiate fixed-price contracts before peak periods instead of buying fully at spot.

**What the model can't do**
The 2017-2018 test period contained a real supply shock — drought in California, Mexican border friction. Prophet captured the seasonal and holiday patterns, but it has no way to predict shocks like these from price history alone. Any production deployment needs a human override layer for supply-side events, or external regressors (weather, import volume, exchange rate) added on top.